<a href="https://colab.research.google.com" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análisis de partido — [EQUIPO_LOCAL] vs [EQUIPO_VISITANTE]
> Completar el header con los datos del partido antes de ejecutar.

**Competición:** ← reemplazar  
**Fecha:** ← reemplazar  
**Resultado:** ← reemplazar  
**Fuente de datos:** FBref  
**Autor:** Andrés  

---

## Estructura del notebook

1. Setup e importaciones  
2. Carga del dataset  
3. Limpieza  
4. EDA — Análisis exploratorio

> Los datos corresponden a las estadísticas individuales de los jugadores del partido.  
> La limpieza se realiza sobre `df_raw` y el análisis sobre `df_clean`.

## 1. Setup

### 1.1 Importaciones y configuración global

**⚠️ Única celda que cambia partido a partido — completar antes de ejecutar.**

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Supresión de warnings
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

# Configuración de visualización del DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configuración de gráficos
sns.set_theme(style='whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 16
plt.rcParams['axes.labelsize'] = 13

# ── CAMBIAR ESTOS VALORES PARTIDO A PARTIDO ──────────────────────────────
EQUIPO_LOCAL     = 'Equipo Local'        # ← reemplazar
EQUIPO_VISITANTE = 'Equipo Visitante'    # ← reemplazar
RESULTADO        = 'X - X'              # ← reemplazar (ej: '1 - 0')
FECHA            = 'DD/MM/AAAA'         # ← reemplazar
COMPETICION      = 'Liga / Copa'         # ← reemplazar
DATA_PATH        = '../data/raw/NOMBRE_ARCHIVO_raw.csv'  # ← reemplazar
# ─────────────────────────────────────────────────────────────────────────

# Rutas de salida — se crean automáticamente
OUTPUT_PATH = '../outputs'
CLEAN_PATH  = '../data/clean'
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(CLEAN_PATH, exist_ok=True)

# Nombre base para archivos de salida
PARTIDO_ID = f"{EQUIPO_LOCAL.lower().replace(' ', '_')}_vs_{EQUIPO_VISITANTE.lower().replace(' ', '_')}"

print('✅ Configuración lista')
print(f'   Partido: {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} | {RESULTADO} | {FECHA}')

### 1.2 Diccionarios de traducción FBref

Traducción de columnas y posiciones — no modificar.

In [ ]:
COLUMNAS_ES = {
    'Player':        'Jugador',
    '#':             'Camiseta',
    'Pos':           'Posición',
    'Pos_Principal': 'Posición principal',
    'Nation':        'País',
    'Min':           'Minutos',
    'Gls':           'Goles',
    'Ast':           'Asistencias',
    'PK':            'Penales convertidos',
    'PKatt':         'Penales intentados',
    'Sh':            'Tiros totales',
    'SoT':           'Tiros al arco',
    'CrdY':          'Tarjetas amarillas',
    'CrdR':          'Tarjetas rojas',
    'Fls':           'Faltas cometidas',
    'Fld':           'Faltas recibidas',
    'Off':           'Fueras de juego',
    'Crs':           'Centros',
    'TklW':          'Entradas ganadas',
    'Int':           'Intercepciones',
    'OG':            'Goles en contra',
    'PKwon':         'Penales ganados',
    'PKcon':         'Penales concedidos',
    'edad_anios':    'Edad (años)',
    'edad_dias':     'Edad (días)',
}

POSICIONES_ES = {
    'GK': 'Arquero',        'DF': 'Defensor',
    'MF': 'Mediocampista',  'FW': 'Delantero',
    'FB': 'Lateral',        'LB': 'Lateral izquierdo',
    'RB': 'Lateral derecho','CB': 'Defensor central',
    'DM': 'Mediocampista defensivo', 'CM': 'Mediocampista central',
    'LM': 'Mediocampista izquierdo', 'RM': 'Mediocampista derecho',
    'WM': 'Mediocampista amplio',    'LW': 'Extremo izquierdo',
    'RW': 'Extremo derecho',         'AM': 'Mediocampista ofensivo',
    'FWMF': 'Delantero/Mediocampista', 'MFFW': 'Mediocampista/Delantero',
    'DFMF': 'Defensor/Mediocampista',  'MFDF': 'Mediocampista/Defensor',
}

def traducir_columnas(df):
    """Renombra columnas FBref al español. Solo traduce las que existen."""
    columnas_presentes = {k: v for k, v in COLUMNAS_ES.items() if k in df.columns}
    df = df.rename(columns=columnas_presentes)
    print(f'Columnas traducidas: {len(columnas_presentes)}')
    return df

def traducir_posiciones(df, columna='Posición'):
    """Traduce abreviaturas de posición. Maneja posiciones combinadas (ej: DF,FW)."""
    if columna not in df.columns:
        print(f"Columna '{columna}' no encontrada")
        return df
    def traducir_pos(valor):
        if pd.isna(valor):
            return valor
        partes = str(valor).split(',')
        traducidas = [POSICIONES_ES.get(p.strip(), p.strip()) for p in partes]
        return ' / '.join(traducidas)
    df[columna] = df[columna].apply(traducir_pos)
    return df

print('✅ Diccionarios y funciones de traducción listos')

### 1.3 Función de perfil de jugador

Reutilizable en cualquier análisis — no modificar.

In [ ]:
def perfil_jugador(df: pd.DataFrame, nombre: str) -> None:
    """
    Muestra el perfil completo de un jugador comparado con el promedio de su posición.
    Indicadores: ▲ por encima | ▼ por debajo | ─ igual al promedio
    """
    jugador = df[df['Jugador'].str.contains(nombre, case=False, na=False)]

    if jugador.empty:
        print(f"Jugador '{nombre}' no encontrado")
        print(f"Jugadores disponibles: {df['Jugador'].tolist()}")
        return

    jugador = jugador.iloc[0]
    posicion = jugador['Posición principal']
    promedio_pos = df[df['Posición principal'] == posicion].select_dtypes(include='number').mean()

    metricas = ['Minutos', 'Tiros totales', 'Tiros al arco', 'Goles',
                'Asistencias', 'Entradas ganadas', 'Intercepciones',
                'Faltas cometidas', 'Faltas recibidas', 'Centros']
    metricas_presentes = [m for m in metricas if m in df.columns]

    print(f"\n{'='*60}")
    print(f"PERFIL: {jugador['Jugador']} | {posicion} | {jugador['Minutos']} min")
    print(f"Partido: {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})")
    print(f"{'='*60}")
    print(f"{'Métrica':<25} {'Valor':>8} {'Prom. posición':>15}  Ind.")
    print(f"{'-'*60}")
    for m in metricas_presentes:
        if m in jugador.index and m in promedio_pos.index:
            diff = jugador[m] - promedio_pos[m]
            ind = '▲' if diff > 0 else ('▼' if diff < 0 else '─')
            print(f"{m:<25} {jugador[m]:>8.0f} {promedio_pos[m]:>15.1f}  {ind}")
    print(f"{'='*60}")
    print('▲ por encima del promedio | ▼ por debajo | ─ igual')

print('✅ Función perfil_jugador lista')

## 2. Carga del dataset

In [ ]:
df_raw = pd.read_csv(DATA_PATH, sep=',', encoding='utf-8')
print(f'✅ Dataset cargado — Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}')
print(f'   Fuente: FBref | Partido: {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} | Fecha: {FECHA}')

## 3. Limpieza

### 3.1 Diagnóstico inicial

Primer vistazo al dataset crudo para identificar problemas de estructura, tipos y calidad.

In [ ]:
print('\n--- Primeras 5 filas ---')
display(df_raw.head())

print('\n--- Últimas 5 filas ---')
display(df_raw.tail())

print(f'\nFilas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}')

print('\n--- Info general ---')
df_raw.info()

print('\n--- Nulos por columna ---')
if df_raw.isnull().sum().sum() == 0:
    print('✅ Sin valores nulos')
else:
    nulos = pd.DataFrame({
        'Nulos': df_raw.isnull().sum(),
        'Porcentaje': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
    })
    display(nulos[nulos['Nulos'] > 0])

print(f'\nFilas duplicadas: {df_raw.duplicated().sum()}')

print('\n--- Estadísticas descriptivas ---')
display(df_raw.describe())

### 3.2 Eliminar columnas técnicas de FBref

FBref incluye columnas que no aportan valor analítico:
- `PKwon` / `PKcon`: penales ganados/concedidos — casi siempre vacías en partidos
- `-9999`: hash técnico interno de FBref

**⚠️ Si el partido tiene datos en PKwon/PKcon, comentar esas líneas.**

In [ ]:
# Columnas a eliminar — ajustar si alguna tiene datos relevantes
cols_eliminar = []

for col in ['PKwon', 'PKcon', '-9999']:
    if col in df_raw.columns:
        cols_eliminar.append(col)

if cols_eliminar:
    df_raw = df_raw.drop(columns=cols_eliminar)
    print(f'✅ Columnas eliminadas: {cols_eliminar}')
else:
    print('ℹ️  No se encontraron columnas técnicas para eliminar')

print(f'   Shape actual: {df_raw.shape}')

### 3.3 Limpiar columna Nation

FBref incluye un código de bandera junto al código de país (ej: `ar ARG`).  
Nos quedamos solo con los últimos 3 caracteres (código ISO del país).

In [ ]:
if 'Nation' in df_raw.columns:
    df_raw['Nation'] = df_raw['Nation'].str[-3:]
    print('✅ Columna Nation normalizada')
    print(df_raw['Nation'].value_counts().to_string())
else:
    print('ℹ️  Columna Nation no encontrada en este dataset')

### 3.4 Separar columna Age

FBref almacena la edad en formato `años-días` (ej: `32-352`).  
Separamos en dos columnas numéricas.

In [ ]:
if 'Age' in df_raw.columns:
    df_raw[['edad_anios', 'edad_dias']] = df_raw['Age'].str.split('-', expand=True).astype(int)

    print(f"NaN en edad_anios: {df_raw['edad_anios'].isnull().sum()}")
    print(f"NaN en edad_dias:  {df_raw['edad_dias'].isnull().sum()}")
    display(df_raw[['Age', 'edad_anios', 'edad_dias']].head(5))

    df_raw = df_raw.drop(columns=['Age'])
    print('✅ Columna Age separada y eliminada')
else:
    print('ℹ️  Columna Age no encontrada — puede que ya esté procesada')

### 3.5 Detección de outliers

Método IQR para identificar valores extremos en variables numéricas.

In [ ]:
def detectar_outliers(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    lim_inf = Q1 - 1.5 * IQR
    lim_sup = Q3 + 1.5 * IQR
    outliers = df[(df[columna] < lim_inf) | (df[columna] > lim_sup)]
    print(f'{columna}:')
    print(f'  Rango normal: [{lim_inf:.2f}, {lim_sup:.2f}]')
    print(f'  Outliers: {len(outliers)} ({len(outliers)/len(df)*100:.1f}%)\n')

columnas_numericas_raw = df_raw.select_dtypes(include=['number']).columns

for col in columnas_numericas_raw:
    detectar_outliers(df_raw, col)
    print('-' * 40)

### 3.6 Crear columna Posición principal

FBref puede asignar posiciones híbridas (ej: `FWMF`).  
Creamos una columna con los primeros 2 caracteres para simplificar el agrupamiento.

**⚠️ Revisar si hay jugadores con posición incorrecta y corregir manualmente abajo.**

In [ ]:
df_raw['Pos_Principal'] = df_raw['Pos'].str[:2]

# ── Correcciones manuales — agregar/quitar según el partido ──────────────
# df_raw.loc[df_raw['Player'] == 'NOMBRE JUGADOR', 'Pos_Principal'] = 'XX'
# ─────────────────────────────────────────────────────────────────────────

print('✅ Posición principal creada')
print(df_raw[['Player', 'Pos', 'Pos_Principal']].to_string())

### 3.7 Traducción de columnas y posiciones

In [ ]:
# Primero traducir posiciones (mientras las columnas aún están en inglés)
df_raw = traducir_posiciones(df_raw, columna='Pos')
df_raw = traducir_posiciones(df_raw, columna='Pos_Principal')

# Después traducir nombres de columnas
df_raw = traducir_columnas(df_raw)

print('✅ Traducción aplicada')
display(df_raw.head())

### 3.8 Tipos de datos finales

In [ ]:
print('--- Tipos de datos ---')
print(df_raw.dtypes)

In [ ]:
# Conversiones necesarias — ajustar según el partido
if 'Edad (días)' in df_raw.columns:
    df_raw['Edad (días)'] = df_raw['Edad (días)'].astype(int)
    print('✅ Edad (días) → int64')

### 3.9 Reset de índice

In [ ]:
print(f'Índice antes del reset: {df_raw.index.tolist()[:10]}...')
df_raw = df_raw.reset_index(drop=True)
print(f'Índice después del reset: {df_raw.index.tolist()[:10]}...')
print(f'Filas finales: {df_raw.shape[0]}')

### 3.10 Reporte de calidad post-limpieza

In [ ]:
print('=' * 55)
print('REPORTE FINAL DE CALIDAD')
print('=' * 55)
print(f'Partido:            {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE}')
print(f'Resultado:          {RESULTADO}')
print(f'Fecha:              {FECHA}')
print(f'Competición:        {COMPETICION}')
print(f'Filas finales:      {df_raw.shape[0]}')
print(f'Columnas:           {df_raw.shape[1]}')
print(f'Duplicados:         {df_raw.duplicated().sum()}')
print(f'Nulos totales:      {df_raw.isnull().sum().sum()}')
print('=' * 55)

### 3.11 Exportar dataset limpio

In [ ]:
df_clean = df_raw.copy()

nombre_salida = f'{CLEAN_PATH}/{PARTIDO_ID}_clean.csv'
df_clean.to_csv(nombre_salida, index=False, encoding='utf-8')
print(f'✅ Dataset exportado: {nombre_salida}')
print(f'   Filas: {df_clean.shape[0]} | Columnas: {df_clean.shape[1]}')

## 4. EDA — Análisis exploratorio

### 4.1 Diagnóstico estructural

In [ ]:
print('--- Info general ---')
df_clean.info()

print('\n--- Estadísticas descriptivas ---')
display(df_clean.describe())

print('\n--- Primeras filas ---')
display(df_clean.head())

print('\n--- Nulos ---')
nulos = pd.DataFrame({
    'Nulos': df_clean.isnull().sum(),
    'Porcentaje': (df_clean.isnull().sum() / len(df_clean) * 100).round(2)
})
if nulos[nulos['Nulos'] > 0].shape[0] > 0:
    display(nulos[nulos['Nulos'] > 0])
else:
    print('✅ Sin nulos')

print(f'\nFilas duplicadas: {df_clean.duplicated().sum()}')

print('\n--- Valores únicos por columna ---')
display(df_clean.nunique().sort_values().to_frame(name='Valores únicos'))

### 4.2 Correlación entre variables numéricas

**Pregunta táctica:** ¿Qué métricas del partido se relacionan entre sí?

In [ ]:
columnas_numericas = df_clean.select_dtypes(include='number').columns.tolist()

plt.figure(figsize=(12, 10))
correlacion = df_clean[columnas_numericas].corr()

sns.heatmap(
    correlacion,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
)
plt.title(f'Correlación entre variables numéricas\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Filtro de jugadores activos

In [ ]:
df_activos = df_clean[df_clean['Minutos'] > 0].copy()
print(f'Jugadores con minutos jugados: {len(df_activos)} de {len(df_clean)} en el dataset')

### 4.4 Tiros totales por jugador

**Pregunta táctica:** ¿Quiénes generaron más situaciones de peligro ofensivo?

In [ ]:
df_ofensivo = df_activos[['Jugador', 'Posición principal', 'Minutos',
                            'Tiros totales', 'Tiros al arco',
                            'Goles', 'Asistencias']].copy()
df_ofensivo = df_ofensivo.sort_values('Tiros totales', ascending=False)

plt.figure(figsize=(12, 6))
p = sns.barplot(
    data=df_ofensivo.head(10),
    x='Jugador', y='Tiros totales',
    hue='Posición principal', dodge=False, palette='Blues_r'
)
for container in p.containers:
    p.bar_label(container, label_type='edge', padding=3)
plt.title(f'Top 10 por tiros totales\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
plt.xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_top_tiros.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Efectividad de tiro

**Pregunta táctica:** ¿Qué tan certeros fueron los jugadores que tiraron al arco?

In [ ]:
df_tiros = df_activos[df_activos['Tiros totales'] > 0].copy()
df_tiros['efectividad_tiro'] = (
    df_tiros['Tiros al arco'] / df_tiros['Tiros totales'] * 100
).round(1)
df_tiros = df_tiros.sort_values('efectividad_tiro', ascending=False)

plt.figure(figsize=(12, 6))
p = sns.barplot(
    data=df_tiros,
    x='Jugador', y='efectividad_tiro',
    hue='Posición principal', dodge=False, palette='Greens_r'
)
for container in p.containers:
    p.bar_label(container, fmt='%.0f%%', label_type='edge', padding=3)
plt.title(f'Efectividad de tiro (% al arco)\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
plt.ylabel('Tiros al arco (%)')
plt.xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_efectividad_tiro.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Contribuciones ofensivas directas (G+A)

**Pregunta táctica:** ¿Quiénes participaron directamente en los goles del partido?

In [ ]:
df_activos['contribucion_ofensiva'] = df_activos['Goles'] + df_activos['Asistencias']
df_contrib = df_activos[df_activos['contribucion_ofensiva'] > 0].sort_values(
    'contribucion_ofensiva', ascending=False
)

if len(df_contrib) == 0:
    print('Sin goles ni asistencias registradas en este partido')
else:
    plt.figure(figsize=(10, 5))
    p = sns.barplot(
        data=df_contrib,
        x='Jugador', y='contribucion_ofensiva',
        hue='Posición principal', dodge=False, palette='Oranges_r'
    )
    for container in p.containers:
        p.bar_label(container, label_type='edge', padding=3)
    plt.title(f'Contribuciones ofensivas (G+A)\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
    plt.xlabel('')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_contribucion_ofensiva.png', dpi=150, bbox_inches='tight')
    plt.show()

### 4.7 Acciones defensivas por jugador

**Pregunta táctica:** ¿Quiénes fueron los más activos defensivamente?

In [ ]:
df_activos['acciones_defensivas'] = (
    df_activos['Entradas ganadas'] + df_activos['Intercepciones']
)

df_def = df_activos[['Jugador', 'Posición principal', 'Minutos',
                       'Entradas ganadas', 'Intercepciones',
                       'acciones_defensivas']].sort_values(
    'acciones_defensivas', ascending=False
)

plt.figure(figsize=(12, 6))
p = sns.barplot(
    data=df_def.head(10),
    x='Jugador', y='acciones_defensivas',
    hue='Posición principal', dodge=False, palette='Reds_r'
)
for container in p.containers:
    p.bar_label(container, label_type='edge', padding=3)
plt.title(f'Top 10 acciones defensivas (entradas + intercepciones)\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
plt.xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_acciones_defensivas.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.8 Acciones defensivas por posición

**Pregunta táctica:** ¿Qué línea cargó más con el trabajo defensivo?

In [ ]:
df_def_pos = df_activos.groupby('Posición principal')[
    ['Entradas ganadas', 'Intercepciones']
].sum().reset_index()

df_def_pos_melt = df_def_pos.melt(
    id_vars='Posición principal',
    value_vars=['Entradas ganadas', 'Intercepciones'],
    var_name='Acción', value_name='Total'
)

plt.figure(figsize=(12, 6))
p = sns.barplot(
    data=df_def_pos_melt,
    x='Posición principal', y='Total',
    hue='Acción', palette='Set2'
)
for container in p.containers:
    p.bar_label(container, label_type='edge', padding=3)
plt.title(f'Acciones defensivas por posición\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
plt.xlabel('')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_defensiva_por_posicion.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.9 Minutos jugados por jugador

**Pregunta táctica:** ¿Cómo distribuyó el técnico los minutos disponibles?

In [ ]:
df_min = df_activos.sort_values('Minutos', ascending=True)

plt.figure(figsize=(10, 8))
p = sns.barplot(
    data=df_min,
    x='Minutos', y='Jugador',
    hue='Posición principal', dodge=False, palette='husl'
)
plt.axvline(45, color='gray', linestyle='--', alpha=0.7, label="Entretiempo (45')")
plt.axvline(90, color='black', linestyle='--', alpha=0.7, label="Tiempo reglamentario (90')")
plt.title(f'Minutos jugados por jugador\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
plt.legend()
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_minutos_jugados.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.10 Métricas per90 — Rendimiento normalizado

**Pregunta táctica:** ¿Quiénes fueron más intensos en relación a su tiempo de juego?

Normalizar por 90 minutos permite comparar titulares y suplentes en igualdad de condiciones.

In [ ]:
df_per90 = df_activos[df_activos['Minutos'] >= 10].copy()

df_per90['tiros_per90']   = (df_per90['Tiros totales'] / df_per90['Minutos'] * 90).round(2)
df_per90['def_per90']     = (df_per90['acciones_defensivas'] / df_per90['Minutos'] * 90).round(2)
df_per90['centros_per90'] = (df_per90['Centros'] / df_per90['Minutos'] * 90).round(2)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Panel izquierdo — tiros per90
df_tiros_p90 = df_per90.sort_values('tiros_per90', ascending=False)
sns.barplot(
    data=df_tiros_p90.head(10),
    x='tiros_per90', y='Jugador',
    hue='Posición principal', dodge=False, palette='Blues_r', ax=axes[0]
)
for container in axes[0].containers:
    axes[0].bar_label(container, fmt='%.2f', label_type='edge', padding=3)
axes[0].set_title('Tiros per90')
axes[0].set_xlabel('Tiros por 90 minutos')
axes[0].legend(title='Posición', fontsize=8)

# Panel derecho — acciones defensivas per90
df_def_p90 = df_per90.sort_values('def_per90', ascending=False)
sns.barplot(
    data=df_def_p90.head(10),
    x='def_per90', y='Jugador',
    hue='Posición principal', dodge=False, palette='Reds_r', ax=axes[1]
)
for container in axes[1].containers:
    axes[1].bar_label(container, fmt='%.2f', label_type='edge', padding=3)
axes[1].set_title('Acciones defensivas per90')
axes[1].set_xlabel('Acciones defensivas por 90 minutos')
axes[1].legend(title='Posición', fontsize=8)

plt.suptitle(f'Rendimiento por 90 minutos\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_metricas_per90.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.11 Resumen por posición

**Pregunta táctica:** ¿Qué aportó cada línea ofensiva y defensivamente?

In [ ]:
metricas_por_pos = df_activos.groupby('Posición principal').agg(
    jugadores=('Jugador', 'count'),
    minutos_prom=('Minutos', 'mean'),
    tiros_total=('Tiros totales', 'sum'),
    tiros_arco_total=('Tiros al arco', 'sum'),
    goles_total=('Goles', 'sum'),
    def_total=('acciones_defensivas', 'sum'),
    faltas_com=('Faltas cometidas', 'sum'),
    faltas_rec=('Faltas recibidas', 'sum'),
).round(1).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.barplot(
    data=metricas_por_pos,
    x='Posición principal', y='tiros_total',
    hue='Posición principal', legend=False, palette='Blues_r', ax=axes[0]
)
for container in axes[0].containers:
    axes[0].bar_label(container, label_type='edge', padding=3)
axes[0].set_title('Tiros totales por posición')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=30)

sns.barplot(
    data=metricas_por_pos,
    x='Posición principal', y='def_total',
    hue='Posición principal', legend=False, palette='Reds_r', ax=axes[1]
)
for container in axes[1].containers:
    axes[1].bar_label(container, label_type='edge', padding=3)
axes[1].set_title('Acciones defensivas por posición')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.suptitle(f'Aporte ofensivo y defensivo por posición\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_aporte_por_posicion.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n--- Tabla resumen completa ---')
display(metricas_por_pos)

### 4.12 Disciplina

**Pregunta táctica:** ¿Hubo jugadores problemáticos o muy foulleados?

In [ ]:
df_disciplina = df_activos[
    (df_activos['Faltas cometidas'] > 0) |
    (df_activos['Faltas recibidas'] > 0) |
    (df_activos['Tarjetas amarillas'] > 0) |
    (df_activos['Tarjetas rojas'] > 0)
].copy().sort_values('Faltas cometidas', ascending=False)

if len(df_disciplina) == 0:
    print('Sin faltas ni tarjetas registradas en este partido')
else:
    df_disc_melt = df_disciplina[['Jugador', 'Faltas cometidas', 'Faltas recibidas']].melt(
        id_vars='Jugador', var_name='Tipo', value_name='Faltas'
    )

    plt.figure(figsize=(12, 6))
    p = sns.barplot(
        data=df_disc_melt,
        x='Jugador', y='Faltas', hue='Tipo',
        palette={'Faltas cometidas': '#e8341a', 'Faltas recibidas': '#1a73e8'}
    )
    for container in p.containers:
        p.bar_label(container, label_type='edge', padding=3)
    plt.title(f'Faltas cometidas vs recibidas\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})')
    plt.xlabel('')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_disciplina.png', dpi=150, bbox_inches='tight')
    plt.show()

tarjetas = df_activos[
    (df_activos['Tarjetas amarillas'] > 0) | (df_activos['Tarjetas rojas'] > 0)
][['Jugador', 'Posición principal', 'Tarjetas amarillas', 'Tarjetas rojas']]

if len(tarjetas) == 0:
    print('Sin tarjetas en este partido')
else:
    print('--- Tarjetas ---')
    display(tarjetas)

### 4.13 Perfil individual de jugador

**⚠️ Cambiar el nombre del jugador según el análisis que querés hacer.**

In [ ]:
# ── Cambiar el nombre del jugador a analizar ─────────────────────────────
JUGADOR_ANALISIS = 'NOMBRE JUGADOR'    # ← reemplazar
# ─────────────────────────────────────────────────────────────────────────

perfil_jugador(df_activos, JUGADOR_ANALISIS)

## 5. Análisis por posición — Gráficos diferenciales

> Esta sección analiza cada línea táctica con las métricas más preponderantes de su rol.  
> Cada subsección filtra los jugadores de esa posición y los compara entre sí.  
> **Uso:** si querés analizar un jugador específico, buscalo en la subsección de su posición.

### 5.1 Setup — Filtros por posición

Define los filtros y métricas específicas de cada línea táctica.

In [ ]:
import numpy as np
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap

# ── Métricas preponderantes por posición ─────────────────────────────────
METRICAS_POR_ROL = {
    'Arquero': {
        'descripcion': 'El arquero es evaluado por su capacidad de evitar goles y cortar centros.',
        'metricas': ['Minutos', 'Tiros al arco', 'Centros', 'Faltas recibidas'],
        'color': '#6c757d',
        'etiqueta_radar': ['Min', 'Tiros al arco recibidos', 'Centros cortados', 'Faltas recibidas']
    },
    'Defensor': {
        'descripcion': 'El defensor es evaluado por su trabajo defensivo: entradas, intercepciones y duelos aéreos.',
        'metricas': ['Minutos', 'Entradas ganadas', 'Intercepciones', 'Faltas cometidas', 'Faltas recibidas', 'Centros'],
        'color': '#1a73e8',
        'etiqueta_radar': ['Min', 'Entradas', 'Intercepciones', 'Faltas com.', 'Faltas rec.', 'Centros']
    },
    'Mediocampista': {
        'descripcion': 'El mediocampista es evaluado por su trabajo en ambas fases: distribución, recuperación y llegada.',
        'metricas': ['Minutos', 'Tiros totales', 'Asistencias', 'Entradas ganadas', 'Intercepciones', 'Centros', 'Faltas recibidas'],
        'color': '#34a853',
        'etiqueta_radar': ['Min', 'Tiros', 'Asistencias', 'Entradas', 'Intercepciones', 'Centros', 'Faltas rec.']
    },
    'Delantero': {
        'descripcion': 'El delantero es evaluado por su capacidad de generar y convertir situaciones de gol.',
        'metricas': ['Minutos', 'Tiros totales', 'Tiros al arco', 'Goles', 'Asistencias', 'Faltas recibidas'],
        'color': '#e8341a',
        'etiqueta_radar': ['Min', 'Tiros tot.', 'Tiros arco', 'Goles', 'Asistencias', 'Faltas rec.']
    },
}

# ── Mapeo de posiciones traducidas a rol simplificado ────────────────────
MAPA_ROL = {
    'Arquero':                    'Arquero',
    'Defensor':                   'Defensor',
    'Defensor central':           'Defensor',
    'Lateral izquierdo':          'Defensor',
    'Lateral derecho':            'Defensor',
    'Mediocampista':              'Mediocampista',
    'Mediocampista defensivo':    'Mediocampista',
    'Mediocampista central':      'Mediocampista',
    'Mediocampista ofensivo':     'Mediocampista',
    'Mediocampista izquierdo':    'Mediocampista',
    'Mediocampista derecho':      'Mediocampista',
    'Mediocampista amplio':       'Mediocampista',
    'Delantero':                  'Delantero',
    'Extremo izquierdo':          'Delantero',
    'Extremo derecho':            'Delantero',
    'Delantero/Mediocampista':    'Delantero',
    'Mediocampista/Delantero':    'Mediocampista',
    'Defensor/Mediocampista':     'Defensor',
    'Mediocampista/Defensor':     'Mediocampista',
}

df_activos['Rol'] = df_activos['Posición principal'].map(MAPA_ROL).fillna('Otro')

print('✅ Setup de posiciones listo')
print('\nDistribución de roles en el partido:')
print(df_activos['Rol'].value_counts().to_string())

### 5.2 Delanteros — Análisis ofensivo

**Pregunta táctica:** ¿Cómo se compararon los delanteros entre sí en las métricas clave de su rol?

Métricas preponderantes: tiros totales, tiros al arco, goles, asistencias, faltas recibidas.

In [ ]:
ROL = 'Delantero'
config = METRICAS_POR_ROL[ROL]
df_rol = df_activos[(df_activos['Rol'] == ROL) & (df_activos['Minutos'] >= 10)].copy()

if len(df_rol) < 2:
    print(f'No hay suficientes {ROL.lower()}s con minutos para comparar ({len(df_rol)} encontrados)')
else:
    metricas = [m for m in config['metricas'] if m in df_rol.columns]
    color_base = config['color']

    fig, axes = plt.subplots(1, 2, figsize=(17, max(5, len(df_rol) * 0.6 + 2)))

    # Panel izq — Lollipop de tiros totales
    df_sorted = df_rol.sort_values('Tiros totales', ascending=True)
    media_tiros = df_rol['Tiros totales'].mean()

    for _, row in df_sorted.iterrows():
        c = color_base if row['Tiros totales'] >= media_tiros else '#adb5bd'
        axes[0].plot([0, row['Tiros totales']], [row['Jugador'], row['Jugador']],
                     color=c, linewidth=2.5, alpha=0.8)
        axes[0].scatter(row['Tiros totales'], row['Jugador'], color=c, s=120, zorder=5)
        axes[0].text(row['Tiros totales'] + 0.1, row['Jugador'],
                     f" {int(row['Tiros totales'])}", va='center', fontsize=9)

    axes[0].axvline(media_tiros, color='gray', linestyle='--', alpha=0.5,
                    linewidth=1, label=f'Prom: {media_tiros:.1f}')
    axes[0].set_title('Tiros totales', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Tiros')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    axes[0].grid(axis='x', alpha=0.2)
    axes[0].legend(fontsize=9)

    # Panel der — Bubble: tiros vs faltas recibidas, tamaño = goles
    df_bubble = df_rol.copy()
    for _, row in df_bubble.iterrows():
        size = max(row['Goles'] * 300 + 80, 80)
        axes[1].scatter(row['Faltas recibidas'], row['Tiros totales'],
                        s=size, color=color_base, alpha=0.75,
                        edgecolors='white', linewidth=1.5)
        axes[1].annotate(row['Jugador'].split()[-1],
                         (row['Faltas recibidas'], row['Tiros totales']),
                         textcoords='offset points', xytext=(6, 4), fontsize=8)

    axes[1].set_xlabel('Faltas recibidas', fontsize=10)
    axes[1].set_ylabel('Tiros totales', fontsize=10)
    axes[1].set_title('Tiros vs Faltas recibidas\n(tamaño = goles)', fontsize=12, fontweight='bold')
    axes[1].text(0.98, 0.02, 'Tamaño = goles convertidos',
                 transform=axes[1].transAxes, ha='right', fontsize=8, color='gray')
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    axes[1].grid(alpha=0.2)

    plt.suptitle(f'Análisis de {ROL}s — {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_delanteros.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n--- Tabla completa: {ROL}s ---')
    display(df_rol[['Jugador', 'Posición principal', 'Minutos'] + metricas[1:]])

### 5.3 Mediocampistas — Análisis de doble fase

**Pregunta táctica:** ¿Cómo se distribuyó el trabajo entre los mediocampistas en ambas fases del juego?

Métricas preponderantes: tiros, asistencias, entradas ganadas, intercepciones, centros, faltas recibidas.

In [ ]:
ROL = 'Mediocampista'
config = METRICAS_POR_ROL[ROL]
df_rol = df_activos[(df_activos['Rol'] == ROL) & (df_activos['Minutos'] >= 10)].copy()

if len(df_rol) < 2:
    print(f'No hay suficientes {ROL.lower()}s con minutos para comparar ({len(df_rol)} encontrados)')
else:
    metricas = [m for m in config['metricas'] if m in df_rol.columns]
    color_base = config['color']

    df_rol['fase_ofensiva']  = df_rol['Tiros totales'] + df_rol['Asistencias'] + df_rol['Centros']
    df_rol['fase_defensiva'] = df_rol['Entradas ganadas'] + df_rol['Intercepciones']

    fig, axes = plt.subplots(1, 2, figsize=(17, max(5, len(df_rol) * 0.6 + 2)))

    # Panel izq — Scatter fase ofensiva vs defensiva con cuadrantes
    med_of  = df_rol['fase_ofensiva'].mean()
    med_def = df_rol['fase_defensiva'].mean()

    for _, row in df_rol.iterrows():
        axes[0].scatter(row['fase_defensiva'], row['fase_ofensiva'],
                        s=max(row['Minutos'] * 1.2, 60),
                        color=color_base, alpha=0.75,
                        edgecolors='white', linewidth=1.5)
        axes[0].annotate(row['Jugador'].split()[-1],
                         (row['fase_defensiva'], row['fase_ofensiva']),
                         textcoords='offset points', xytext=(6, 4), fontsize=8)

    axes[0].axvline(med_def, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    axes[0].axhline(med_of,  color='gray', linestyle='--', alpha=0.5, linewidth=1)

    xlim = axes[0].get_xlim(); ylim = axes[0].get_ylim()
    axes[0].text(xlim[1]*0.8, ylim[1]*0.92, 'Box to box', fontsize=8,
                 color='#34a853', fontweight='bold', ha='center')
    axes[0].text(xlim[1]*0.8, ylim[0]+0.1, 'Defensivo puro', fontsize=8,
                 color='#1a73e8', fontweight='bold', ha='center')
    axes[0].text(xlim[0]+0.05, ylim[1]*0.92, 'Creativo', fontsize=8,
                 color='#e8341a', fontweight='bold', ha='center')
    axes[0].text(xlim[0]+0.05, ylim[0]+0.1, 'Bajo impacto', fontsize=8,
                 color='#adb5bd', fontweight='bold', ha='center')

    axes[0].set_xlabel('Fase defensiva (entradas + intercepciones)', fontsize=10)
    axes[0].set_ylabel('Fase ofensiva (tiros + asistencias + centros)', fontsize=10)
    axes[0].set_title('Perfil ofensivo vs defensivo\n(tamaño = minutos jugados)',
                       fontsize=12, fontweight='bold')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    axes[0].grid(alpha=0.2)

    # Panel der — Lollipop acciones defensivas
    df_sorted = df_rol.sort_values('fase_defensiva', ascending=True)
    media_def = df_rol['fase_defensiva'].mean()

    for _, row in df_sorted.iterrows():
        c = color_base if row['fase_defensiva'] >= media_def else '#adb5bd'
        axes[1].plot([0, row['fase_defensiva']], [row['Jugador'], row['Jugador']],
                     color=c, linewidth=2.5, alpha=0.8)
        axes[1].scatter(row['fase_defensiva'], row['Jugador'], color=c, s=120, zorder=5)
        axes[1].text(row['fase_defensiva'] + 0.05, row['Jugador'],
                     f" {int(row['fase_defensiva'])}", va='center', fontsize=9)

    axes[1].axvline(media_def, color='gray', linestyle='--', alpha=0.5,
                    linewidth=1, label=f'Prom: {media_def:.1f}')
    axes[1].set_title('Acciones defensivas totales', fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Entradas + Intercepciones')
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    axes[1].grid(axis='x', alpha=0.2)
    axes[1].legend(fontsize=9)

    plt.suptitle(f'Análisis de {ROL}s — {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_mediocampistas.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n--- Tabla completa: {ROL}s ---')
    display(df_rol[['Jugador', 'Posición principal', 'Minutos'] + metricas[1:]])

### 5.4 Defensores — Análisis defensivo

**Pregunta táctica:** ¿Quiénes fueron los defensores más activos y cuál fue su perfil de intervención?

Métricas preponderantes: entradas ganadas, intercepciones, faltas cometidas, faltas recibidas, centros.

In [ ]:
ROL = 'Defensor'
config = METRICAS_POR_ROL[ROL]
df_rol = df_activos[(df_activos['Rol'] == ROL) & (df_activos['Minutos'] >= 10)].copy()

if len(df_rol) < 2:
    print(f'No hay suficientes {ROL.lower()}es con minutos para comparar ({len(df_rol)} encontrados)')
else:
    metricas = [m for m in config['metricas'] if m in df_rol.columns]
    color_base = config['color']

    df_rol['acciones_def'] = df_rol['Entradas ganadas'] + df_rol['Intercepciones']

    fig, axes = plt.subplots(1, 2, figsize=(17, max(5, len(df_rol) * 0.6 + 2)))

    # Panel izq — Lollipop acciones defensivas totales
    df_sorted = df_rol.sort_values('acciones_def', ascending=True)
    media_def = df_rol['acciones_def'].mean()

    for _, row in df_sorted.iterrows():
        c = color_base if row['acciones_def'] >= media_def else '#adb5bd'
        axes[0].plot([0, row['acciones_def']], [row['Jugador'], row['Jugador']],
                     color=c, linewidth=2.5, alpha=0.8)
        axes[0].scatter(row['acciones_def'], row['Jugador'], color=c, s=120, zorder=5)
        axes[0].text(row['acciones_def'] + 0.05, row['Jugador'],
                     f" {int(row['acciones_def'])}", va='center', fontsize=9)

    axes[0].axvline(media_def, color='gray', linestyle='--', alpha=0.5,
                    linewidth=1, label=f'Prom: {media_def:.1f}')
    axes[0].set_title('Acciones defensivas totales\n(entradas + intercepciones)',
                       fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Acciones defensivas')
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    axes[0].grid(axis='x', alpha=0.2)
    axes[0].legend(fontsize=9)

    # Panel der — Bubble: entradas vs intercepciones, tamaño = centros
    for _, row in df_rol.iterrows():
        size = max(row['Centros'] * 200 + 80, 80)
        axes[1].scatter(row['Entradas ganadas'], row['Intercepciones'],
                        s=size, color=color_base, alpha=0.75,
                        edgecolors='white', linewidth=1.5)
        axes[1].annotate(row['Jugador'].split()[-1],
                         (row['Entradas ganadas'], row['Intercepciones']),
                         textcoords='offset points', xytext=(6, 4), fontsize=8)

    axes[1].set_xlabel('Entradas ganadas', fontsize=10)
    axes[1].set_ylabel('Intercepciones', fontsize=10)
    axes[1].set_title('Entradas vs Intercepciones\n(tamaño = centros realizados)',
                       fontsize=12, fontweight='bold')
    axes[1].text(0.98, 0.02, 'Tamaño = centros',
                 transform=axes[1].transAxes, ha='right', fontsize=8, color='gray')
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    axes[1].grid(alpha=0.2)

    plt.suptitle(f'Análisis de {ROL}es — {EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_PATH}/{PARTIDO_ID}_defensores.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n--- Tabla completa: {ROL}es ---')
    display(df_rol[['Jugador', 'Posición principal', 'Minutos'] + metricas[1:]])

### 5.5 Heatmap táctico por posición

**Pregunta táctica:** ¿Cuál fue el perfil completo de cada jugador dentro de su línea táctica?

Cada heatmap muestra solo las métricas relevantes para ese rol, con valores normalizados dentro del grupo.  
Permite identificar de un vistazo quién hizo más de cada cosa dentro de su posición.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

cmap_custom = LinearSegmentedColormap.from_list(
    'futbol', ['#f8f9fa', '#74c0fc', '#1971c2', '#0b3d91']
)

for rol, config in METRICAS_POR_ROL.items():
    df_rol = df_activos[
        (df_activos['Rol'] == rol) & (df_activos['Minutos'] >= 10)
    ].copy()

    if len(df_rol) < 2:
        print(f'ℹ️  {rol}: menos de 2 jugadores con minutos — se omite el heatmap')
        continue

    metricas = [m for m in config['metricas'] if m in df_rol.columns]
    df_heat = df_rol[['Jugador'] + metricas].set_index('Jugador')

    # Normalizar por columna (0-1) dentro del grupo
    df_norm = df_heat.copy().astype(float)
    for col in df_norm.columns:
        rng = df_norm[col].max() - df_norm[col].min()
        df_norm[col] = (df_norm[col] - df_norm[col].min()) / rng if rng > 0 else 0

    fig, ax = plt.subplots(figsize=(max(8, len(metricas) * 1.3), max(4, len(df_rol) * 0.6 + 1.5)))

    sns.heatmap(
        df_norm,
        annot=df_heat.astype(int),
        fmt='d',
        cmap=cmap_custom,
        linewidths=0.5,
        linecolor='white',
        ax=ax,
        cbar_kws={'label': 'Valor normalizado dentro del grupo'}
    )
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.tick_params(axis='x', rotation=35, labelsize=9)
    ax.tick_params(axis='y', rotation=0, labelsize=9)

    plt.title(
        f'Heatmap — {rol}s\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})\n'
        f'{config["descripcion"]}',
        fontsize=11, fontweight='bold', pad=12
    )
    plt.tight_layout()
    nombre_archivo = f'{OUTPUT_PATH}/{PARTIDO_ID}_heatmap_{rol.lower().replace(" ", "_")}.png'
    plt.savefig(nombre_archivo, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {nombre_archivo}')

### 5.6 Radar chart comparativo por posición

**Pregunta táctica:** ¿Cómo se comparan los jugadores de la misma posición en todas sus métricas clave?

El radar permite ver múltiples dimensiones de cada jugador simultáneamente.  
Los valores se normalizan por percentil dentro del grupo de su posición.

In [ ]:
def radar_por_rol(df_rol: pd.DataFrame, metricas: list, rol: str, color_base: str) -> None:
    """
    Genera un radar chart comparativo para todos los jugadores de un mismo rol.
    Los valores se normalizan por percentil dentro del grupo.
    """
    if len(df_rol) < 2:
        print(f'No hay suficientes jugadores de {rol} para el radar')
        return

    metricas_ok = [m for m in metricas if m in df_rol.columns]

    # Normalizar por percentil dentro del grupo
    df_norm = df_rol.copy()
    for m in metricas_ok:
        df_norm[m] = df_norm[m].rank(pct=True) * 100

    N = len(metricas_ok)
    angulos = [n / float(N) * 2 * np.pi for n in range(N)]
    angulos += angulos[:1]

    colores = ['#e8341a', '#1a73e8', '#34a853', '#fbbc04', '#9c27b0',
               '#00bcd4', '#ff5722', '#607d8b', '#795548', '#009688']

    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

    for idx, (_, row) in enumerate(df_rol.iterrows()):
        fila_norm = df_norm[df_norm['Jugador'] == row['Jugador']].iloc[0]
        valores = [fila_norm[m] if m in fila_norm.index else 0 for m in metricas_ok]
        valores += valores[:1]
        color = colores[idx % len(colores)]
        ax.plot(angulos, valores, linewidth=2, linestyle='solid', color=color,
                label=row['Jugador'])
        ax.fill(angulos, valores, alpha=0.1, color=color)

    ax.set_xticks(angulos[:-1])
    ax.set_xticklabels(metricas_ok, size=9, fontweight='bold')
    ax.set_ylim(0, 100)
    ax.set_yticks([25, 50, 75, 100])
    ax.set_yticklabels(['P25', 'P50', 'P75', 'P100'], size=7, color='gray')
    ax.grid(color='gray', alpha=0.3)
    ax.spines['polar'].set_visible(False)

    plt.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9,
               title=f'{rol}s en el partido')
    plt.title(
        f'Radar comparativo — {rol}s\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})\n'
        f'(valores en percentil dentro del grupo)',
        size=11, pad=20, fontweight='bold'
    )
    plt.tight_layout()
    nombre_archivo = f'{OUTPUT_PATH}/{PARTIDO_ID}_radar_{rol.lower().replace(" ", "_")}.png'
    plt.savefig(nombre_archivo, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {nombre_archivo}')


# Generar radar para cada rol con suficientes jugadores
for rol, config in METRICAS_POR_ROL.items():
    df_rol = df_activos[
        (df_activos['Rol'] == rol) & (df_activos['Minutos'] >= 10)
    ].copy()

    if len(df_rol) >= 2:
        radar_por_rol(df_rol, config['metricas'], rol, config['color'])
    else:
        print(f'ℹ️  {rol}: menos de 2 jugadores con minutos — se omite el radar')

### 5.7 Ranking per90 por posición

**Pregunta táctica:** ¿Quién fue el más destacado dentro de cada línea, normalizando por minutos jugados?

Este ranking es el más justo para comparar titulares con suplentes dentro de la misma posición.

In [ ]:
for rol, config in METRICAS_POR_ROL.items():
    df_rol = df_activos[
        (df_activos['Rol'] == rol) & (df_activos['Minutos'] >= 10)
    ].copy()

    if len(df_rol) < 2:
        continue

    metricas = [m for m in config['metricas'][1:] if m in df_rol.columns]  # excluir Minutos

    # Calcular per90 de cada métrica
    for m in metricas:
        df_rol[f'{m}_per90'] = (df_rol[m] / df_rol['Minutos'] * 90).round(2)

    metricas_per90 = [f'{m}_per90' for m in metricas]

    # Score compuesto: promedio de percentiles per90
    df_norm = df_rol[metricas_per90].copy()
    for col in df_norm.columns:
        rng = df_norm[col].max() - df_norm[col].min()
        df_norm[col] = (df_norm[col] - df_norm[col].min()) / rng * 100 if rng > 0 else 50
    df_rol['score_per90'] = df_norm.mean(axis=1).round(1)

    df_ranking = df_rol[['Jugador', 'Posición principal', 'Minutos', 'score_per90']].sort_values(
        'score_per90', ascending=True
    )

    fig, ax = plt.subplots(figsize=(10, max(4, len(df_rol) * 0.55 + 1.5)))

    colores_bar = [config['color'] if s >= df_ranking['score_per90'].mean() else '#adb5bd'
                   for s in df_ranking['score_per90']]

    bars = ax.barh(df_ranking['Jugador'], df_ranking['score_per90'],
                   color=colores_bar, edgecolor='white', height=0.6)
    for bar, val in zip(bars, df_ranking['score_per90']):
        ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}', va='center', fontsize=9, fontweight='bold')

    ax.axvline(df_ranking['score_per90'].mean(), color='gray',
               linestyle='--', alpha=0.6, linewidth=1,
               label=f"Prom: {df_ranking['score_per90'].mean():.1f}")
    ax.set_xlabel('Score compuesto per90 (0-100)', fontsize=10)
    ax.set_title(
        f'Ranking per90 — {rol}s\n{EQUIPO_LOCAL} vs {EQUIPO_VISITANTE} — {RESULTADO} ({FECHA})\n'
        f'Score = promedio de percentiles de: {", ".join(metricas)}',
        fontsize=11, fontweight='bold'
    )
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='x', alpha=0.2)
    ax.legend(fontsize=9)

    plt.tight_layout()
    nombre_archivo = f'{OUTPUT_PATH}/{PARTIDO_ID}_ranking_{rol.lower().replace(" ", "_")}.png'
    plt.savefig(nombre_archivo, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Guardado: {nombre_archivo}')